# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) Python library in an end-to-end, reproducible workflow.

### Dataset Source
The dataset schema is provided via a [Croissant JSON-LD schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and explore its structure with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)


### Metadata Summary
| Key Attribute         | Value                                                                 |
|----------------------|-----------------------------------------------------------------------|
| Identifier           | 10.71728/senscience.qs2f-h81p                                         |
| License              | https://opendatacommons.org/licenses/by/1-0/                          |
| Version              | 1.0.0                                                                 |
| Number of Authors    | 5                                                                     |
| Data Collection      | 2019-05-01/2025-03-01                                                 |
| Number of Records    | 77                                                                    |

## 2. Data Overview
Discover available record sets, their `@id`s, and field `@id`s in the dataset.

In [ ]:
# List all record sets in the dataset
print("Available record sets:")
for rs in dataset.record_sets:
    print(f"  name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  fields:")
    for field in rs.fields:
        print(f"    {field.name} (@id: {field.id})")
    print('-' * 40)

# For clarity, collect record_set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]

#### Quick Example: Inspect the first record set
Records are loaded as dictionaries mapping field `@id` to values.

In [ ]:
# Preview one record from the first record set (if present)
if record_set_ids:
    print(f"\nExample record from record set: {record_set_ids[0]}")
    for i, record in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(record)
        if i > 0:  # Just show two records
            break
else:
    print("No record sets found in the dataset schema.")

## 3. Data Extraction
Extract all records for each record set into DataFrames for analysis (using only `@id` values as references).

In [ ]:
# Build a DataFrame per record set, using @id strings
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '{record_set_id}'")
    print(f"Columns (@id): {df.columns.tolist()}")
    print('-' * 60)

#### Peek at the main tabular dataset
<br/>
_Replace the `primary_record_set_id` definition below with the most relevant one from your data overview (above) if needed._

In [ ]:
# Pick the main (tabular) record set for further analysis
primary_record_set_id = record_set_ids[0] if record_set_ids else None

if primary_record_set_id:
    print(f"Head of data for record set: {primary_record_set_id}")
    display(dataframes[primary_record_set_id].head())
else:
    print("No record set available for display.")

## 4. Exploratory Data Analysis (EDA)
Apply processing steps like filtering, normalization, aggregation, and grouping. **Only refer to fields/columns by their `@id`**.

_Below, we demonstrate with a numeric field and a group field. Update the variables to match the actual `@id` strings from your record set as needed._

In [ ]:
# Example: identify a numeric and group field by @id -- modify as appropriate for your dataset.
df = dataframes[primary_record_set_id]

# List available columns
print("Available @id columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}: {col}")

# Example field IDs (replace these as appropriate):
# E.g., suppose '@id' for 'Age at Second CRC diagnosis' is 'http://senscience.ai/age_at_diagnosis_2',
# and group field '@id' for 'Sex' is 'http://senscience.ai/sex'
numeric_field_id = df.columns[0] if len(df.columns) > 0 else None  # change if necessary
group_field_id = df.columns[1] if len(df.columns) > 1 else None  # change if necessary

print(f"\nUsing numeric field: {numeric_field_id}")
print(f"Using group field:  {group_field_id}")

# Ensure data types
if numeric_field_id is not None and numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
else:
    print("No valid numeric field ID found. Please update numeric_field_id variable.")

# EDA: Remove outliers and normalize
threshold = df[numeric_field_id].mean() + 2 * df[numeric_field_id].std() if numeric_field_id else None
if threshold is not None:
    filtered_df = df[df[numeric_field_id] < threshold]
    print(f"Filtered records with {numeric_field_id} < {threshold:.1f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized column '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping (if group field exists)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No valid numeric field found for EDA. Please verify the field selection above.")

## 5. Visualization
Create plots to visualize distributions or relationships in the dataset, referencing only `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded rich metadata and records from a **FAIR²-compliant Croissant schema** using `mlcroissant`;
- Explored record sets and referenced all fields by their unique `@id` values;
- Extracted, filtered, and transformed the data for analysis;
- Visualized field distributions and group differences.

This approach ensures that data handling is transparent, reproducible, and fully aligned with emerging FAIR/Linked Data principles.

> **Next steps**: Extend this template with domain-specific analyses or machine learning workflows using the extracted DataFrames. Be sure to always reference entities by their `@id` for maximum interoperability.